In [0]:
csv_filename = "patients.csv"
df_allergies = spark.read.csv(f"{dest_path}/{csv_filename}", header=True, inferSchema=True)
df_allergies.printSchema()
df_allergies.show(10)

In [0]:
csv_filename = "encounters.csv"
df = spark.read.csv(f"{dest_path}/{csv_filename}", header=True, inferSchema=True)
df.printSchema()
df.show(10)

In [0]:
csv_filename = "conditions.csv"
df = spark.read.csv(f"{dest_path}/{csv_filename}", header=True, inferSchema=True)
df.printSchema()
df.show(10)

In [0]:
source_dir = "/Volumes/main/dev_ingridstudien_healthcare_claims_dev/healthcare_claims_landing/"
target_base_dir = "/Volumes/healthcare_dev/00_landing/raw_landing_volume/synthea/"

# List all landing files
files = dbutils.fs.ls(source_dir)

for file in files:
    if file.name.endswith(".csv"):
        # Extract entity name (e.g., 'patients.csv' -> 'patients')
        entity_name = file.name.replace(".csv", "")
        
        # Define destination path with subfolder per entity
        dest_folder = f"{target_base_dir}{entity_name}/"
        dest_path = f"{dest_folder}{file.name}"
        
        # Create entity folder and copy file
        dbutils.fs.mkdirs(dest_folder)
        dbutils.fs.cp(file.path, dest_path)
        print(f"Copied {file.name} -> {dest_path}")

In [0]:
source_dir = "/Volumes/healthcare_dev/00_landing/raw_landing_volume/synthea/"

# List all items inside the synthea directory
items = dbutils.fs.ls(source_dir)

for item in items:
    # If the item is a folder (entity directory like patients/, allergies/)
    if item.isDir():
        sub_files = dbutils.fs.ls(item.path)
        for sub_file in sub_files:
            if sub_file.name.endswith(".csv"):
                # Read CSV with Spark
                df = spark.read.csv(sub_file.path, header=True)
                # Display entity name, file name, and total row count
                print(f"Entity: {item.name.strip('/')} | File: {sub_file.name} | Rows: {df.count()}")
                
    # If the CSV files are sitting directly in the root directory
    elif item.name.endswith(".csv"):
        df = spark.read.csv(item.path, header=True)
        print(f"File: {item.name} | Rows: {df.count()}")

In [0]:
source_dir = "/Volumes/healthcare_dev/00_landing/raw_landing_volume/synthea/"

# List all items inside the synthea directory
items = dbutils.fs.ls(source_dir)

for item in items:
    # If the item is a folder (entity directory like patients/, allergies/)
    if item.isDir():
        sub_files = dbutils.fs.ls(item.path)
        for sub_file in sub_files:
            if sub_file.name.endswith(".csv"):
                # Read CSV with Spark
                df = spark.read.csv(sub_file.path, header=True)
                # Display entity name, file name, and total row count
                print(item.name.strip('/'))
                
    # If the CSV files are sitting directly in the root directory
    elif item.name.endswith(".csv"):
        df = spark.read.csv(item.path, header=True)
        print(f"File: {item.name} | Rows: {df.count()}")

In [0]:
%sql
USE CATALOG healthcare_dev;
USE SCHEMA 00_landing;
SHOW VOLUMES